# Scan-phase benchmark - current `mbo_utilities`

Per-window-size benchmark of `bidir_phasecorr`. Run this in `C:\Users\RBO\repos\mbo_utilities\.venv` kernel.

Sibling: `scanphase_v2-7-7.ipynb` runs the identical body in the v2-7-7 venv. Both save JSON to `D:\scanphase_versions\` so a third notebook can compare cross-version.

- **Window sizes**: 1, 10, 100, 200, 500 frames (always the FIRST N frames so both versions hit identical bytes)
- **Variants**: `use_fft=False` (integer-precision) and `use_fft=True` (FFT subpixel)
- **Repeats**: 5 timed runs per cell, report median (plus min/max)
- **Method**: `mean` reduction (one offset per window, computed from the mean image)
- **Bonus**: per-frame offsets across all 500 frames - measures frame-to-frame stability (proxy for accuracy)

In [1]:
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import time

import numpy as np
import mbo_utilities as mbo
from mbo_utilities.analysis.phasecorr import bidir_phasecorr

VERSION_LABEL = "current"
INPUT = Path(r"D:\demo\raw")
SAVE_DIR = Path(r"D:\scanphase_versions")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = SAVE_DIR / f"{VERSION_LABEL}.json"

WINDOW_SIZES = [1, 10, 100, 200, 500]
USE_FFT_VARIANTS = [False, True]
REPEATS = 5
PLANE_INDEX = 0          # 0-indexed - corresponds to the suite2p plane=1 from earlier runs
CHANNEL_INDEX = 0

def _v(pkg):
    try: return version(pkg)
    except PackageNotFoundError: return None

PKG_VERSIONS = {p: _v(p) for p in ("mbo_utilities", "numpy", "scipy", "scikit-image")}
for k, v in PKG_VERSIONS.items():
    print(f"{k:<18} {v}")

mbo_utilities      3.0.0
numpy              2.3.5
scipy              1.17.1
scikit-image       0.26.0


In [2]:
# load lazy array; force fix_phase=False so we receive raw, uncorrected frames
arr = mbo.imread(INPUT)
if hasattr(arr, "fix_phase"):
    arr.fix_phase = False
if hasattr(arr, "use_fft"):
    arr.use_fft = False
print(f"arr.shape       = {arr.shape}")
print(f"arr.fix_phase   = {getattr(arr, 'fix_phase', None)}")
print(f"arr.use_fft     = {getattr(arr, 'use_fft', None)}")

# materialize one (max(WINDOW_SIZES), Y, X) chunk so every variant sees the
# exact same bytes; pulls plane_index, channel_index out of TCZYX layout.
max_win = max(WINDOW_SIZES)
chunk = arr[:max_win, CHANNEL_INDEX, PLANE_INDEX, :, :]
if hasattr(chunk, "compute"):
    chunk = chunk.compute()
chunk = np.asarray(chunk).astype(np.int16)
print(f"chunk.shape     = {chunk.shape}  dtype={chunk.dtype}")

Counting frames:   0%|          | 0/2 [00:00<?, ?it/s]

arr.shape       = (1574, 1, 14, 550, 448)
arr.fix_phase   = False
arr.use_fft     = False
chunk.shape     = (500, 550, 448)  dtype=int16


In [3]:
def fmt_offset(off):
    a = np.asarray(off)
    if a.ndim == 0:
        return f"{float(a):.4f}"
    return f"[{a.size} vals: mean={a.mean():.4f}, std={a.std():.4f}]"

def offset_to_json(off):
    a = np.asarray(off)
    if a.ndim == 0:
        return float(a)
    return [float(x) for x in a.ravel()]

results = []
for use_fft in USE_FFT_VARIANTS:
    for win in WINDOW_SIZES:
        frames = chunk[:win]                                           # (win, Y, X)
        # warmup (excluded) - first call may pay FFT planning / JIT cost
        bidir_phasecorr(frames, method="mean", use_fft=use_fft)
        # timed runs
        times_ms = []
        last_off = None
        for _ in range(REPEATS):
            t0 = time.perf_counter()
            _, last_off = bidir_phasecorr(frames, method="mean", use_fft=use_fft)
            times_ms.append((time.perf_counter() - t0) * 1000.0)
        rec = {
            "window_size": win,
            "use_fft": bool(use_fft),
            "offset": offset_to_json(last_off),
            "time_ms_median": float(np.median(times_ms)),
            "time_ms_min": float(np.min(times_ms)),
            "time_ms_max": float(np.max(times_ms)),
            "time_ms_all": [float(x) for x in times_ms],
        }
        results.append(rec)
        print(f"use_fft={int(use_fft)}  win={win:>4d}  offset={fmt_offset(last_off):<24s}  median={rec['time_ms_median']:>8.2f} ms  (min {rec['time_ms_min']:.2f}, max {rec['time_ms_max']:.2f})")

use_fft=0  win=   1  offset=1.0000                    median=    4.47 ms  (min 3.92, max 5.04)
use_fft=0  win=  10  offset=1.0000                    median=    4.25 ms  (min 4.21, max 4.96)
use_fft=0  win= 100  offset=1.0000                    median=   30.55 ms  (min 30.39, max 31.43)
use_fft=0  win= 200  offset=1.0000                    median=   58.03 ms  (min 56.74, max 64.85)
use_fft=0  win= 500  offset=1.0000                    median=  144.00 ms  (min 136.65, max 145.30)
use_fft=1  win=   1  offset=0.8404                    median=    2.72 ms  (min 2.69, max 2.81)
use_fft=1  win=  10  offset=0.7705                    median=   16.90 ms  (min 15.92, max 16.94)
use_fft=1  win= 100  offset=0.9781                    median=  149.85 ms  (min 149.06, max 163.23)
use_fft=1  win= 200  offset=0.9495                    median=  295.17 ms  (min 290.10, max 303.05)
use_fft=1  win= 500  offset=0.9568                    median=  762.80 ms  (min 746.24, max 771.02)


In [4]:
# per-frame estimator on all 500 frames - measures frame-to-frame noise
for use_fft in USE_FFT_VARIANTS:
    bidir_phasecorr(chunk, method="frame", use_fft=use_fft)              # warmup
    t0 = time.perf_counter()
    _, offs = bidir_phasecorr(chunk, method="frame", use_fft=use_fft)
    elapsed_ms = (time.perf_counter() - t0) * 1000.0
    a = np.asarray(offs).astype(float)
    rec = {
        "window_size": "per_frame",
        "use_fft": bool(use_fft),
        "per_frame_offsets": a.tolist(),
        "per_frame_mean": float(a.mean()),
        "per_frame_std": float(a.std()),
        "per_frame_min": float(a.min()),
        "per_frame_max": float(a.max()),
        "time_ms_total": elapsed_ms,
        "time_ms_per_frame": elapsed_ms / len(a),
        "n_frames": int(len(a)),
    }
    results.append(rec)
    print(f"per_frame  use_fft={int(use_fft)}  n={len(a)}  mean={a.mean():.4f}  std={a.std():.4f}  range=[{a.min():.4f}, {a.max():.4f}]  total={elapsed_ms:.1f} ms  ({elapsed_ms/len(a):.2f} ms/frame)")

per_frame  use_fft=0  n=500  mean=0.9300  std=0.3051  range=[0.0000, 2.0000]  total=920.8 ms  (1.84 ms/frame)
per_frame  use_fft=1  n=500  mean=0.8521  std=0.2070  range=[0.2170, 1.6942]  total=1190.3 ms  (2.38 ms/frame)


In [5]:
payload = {
    "version_label": VERSION_LABEL,
    "package_versions": PKG_VERSIONS,
    "input_path": str(INPUT),
    "plane_index": PLANE_INDEX,
    "channel_index": CHANNEL_INDEX,
    "chunk_shape": list(chunk.shape),
    "window_sizes": WINDOW_SIZES,
    "use_fft_variants": USE_FFT_VARIANTS,
    "repeats": REPEATS,
    "results": results,
}
with open(OUT_PATH, "w") as f:
    json.dump(payload, f, indent=2)
print(f"wrote {OUT_PATH} ({OUT_PATH.stat().st_size:,d} bytes)")

wrote D:\scanphase_versions\current.json (26,643 bytes)
